In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from pymongo import MongoClient
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.ar_model import AutoReg
import time
import matplotlib.pyplot as plt


## Reading the data

In [2]:
def wrangle(collection, resample_freq="1h"):

    # Fetch data 
    results = collection.find(
        {}, 
        projection={"datetimeLocal": 1, "value": 1, "_id": 0}
    )
    
    df = pd.DataFrame(list(results))
    
    #rename columns
    df = df.rename(columns={"datetimeLocal": "timestamp", "value": "pm25"})
    #convert datatype
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    
    # Set index and handle timezone
    df.set_index("timestamp", inplace=True)
    
    df.index = df.index.tz_convert("Africa/Casablanca")

    # Remove outliers 
    df = df[df["pm25"] < 500]

    # Resample to 1-hour window, forward-fill missing values
    y = df["pm25"].resample(resample_freq).mean().interpolate()


    return y

In [8]:
client = MongoClient(
            host="localhost",
            port=27017,
            username="admin",
            password="admin")
db=client["air-quality"]
morraco = db["morraco"]

In [9]:
y = wrangle(morraco)

# Spliting Data

In [13]:
cutoff_test = int(len(y) * 0.95)
y_train = y.iloc[:cutoff_test]
y_test =  y.iloc[cutoff_test:]

## Grid search

In [16]:
p_params = range(1,26,8)
q_params = range(1,3)

In [17]:
# Create dictionary to store MAEs
mae_grid = dict()
# Outer loop: Iterate through possible values for `p`
for p in p_params:
    # Create key-value pair in dict. Key is `p`, value is empty list.
    mae_grid[p] = list()
    # Inner loop: Iterate through possible values for `q`
    for q in q_params:
        # Combination of hyperparameters for model
        order = (p, 0, q)
        # Note start time
        start_time = time.time()
        # Train model
        model = ARIMA(y_train,order=order).fit()
        # Calculate model training time
        elapsed_time = round(time.time() - start_time, 2)
        print(f"Trained ARIMA {order} in {elapsed_time} seconds.")
        # Generate in-sample (training) predictions
        y_pred = model.predict()
        # Calculate training MAE
        mae = mean_absolute_error(y_train, y_pred)
        # Append MAE to list in dictionary
        mae_grid[p].append(mae)

print()
print(mae_grid)

Trained ARIMA (1, 0, 1) in 0.32 seconds.
Trained ARIMA (1, 0, 2) in 0.46 seconds.
Trained ARIMA (9, 0, 1) in 3.08 seconds.
Trained ARIMA (9, 0, 2) in 2.89 seconds.
Trained ARIMA (17, 0, 1) in 9.61 seconds.
Trained ARIMA (17, 0, 2) in 7.28 seconds.
Trained ARIMA (25, 0, 1) in 26.7 seconds.


c:\Users\Muhammed\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Trained ARIMA (25, 0, 2) in 40.83 seconds.

{1: [5.5587444048995005, 5.556550544025811], 9: [5.523768932049746, 5.524365581583662], 17: [5.524884873893833, 5.5259515144278115], 25: [5.399755913589206, 5.389101772017279]}


In [18]:
mae_df = pd.DataFrame(mae_grid)
mae_df.round(4)

,1,9,17,25
0,5.5587,5.5238,5.5249,5.3998
1,5.5566,5.5244,5.5260,5.3891
